# Two-Tower Model (TTN)

Build a PyTorch two-tower model on the Home & Kitchen interactions and the
extracted item features. See `README.md` in this folder for the design.

BPR-style pairwise ranking loss: `-log σ(score(u, i) - score(u, j))`.

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

## Load data

Two sources, both keyed on `asin`:

- **Item features** — `data/df_features.pkl`: one row per item (~1.13M items)
  with the extracted attributes (`cat_*`, `brand`, `Product_Type`,
  `Material`, `Color`, ...) plus `title_cleaned`.
- **Comments / reviews** — `data/Home_and_Kitchen_filtered.csv`: one row per
  review (the interactions), with `reviewerID`, `asin`, `overall`,
  `unixReviewTime`, etc.

We connect them with a left join of the reviews onto the item features so
each interaction row also carries its item's extracted features.

In [3]:
from pathlib import Path

# data/ lives at the repo root, one level up from this ttn/ folder
DATA_DIR = Path("..") / "data"

# --- Item features (one row per asin) ---
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")
print("df_features:", df_features.shape)
df_features.head(3)

df_features: (1134566, 81)


,category,tech1,description,title,tech2,brand,feature,rank,main_cat,price,...,dimension_unit,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,density_weight_lb_cleaned,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,voltage_numeric_cleaned,thread_count_numeric_cleaned,weight_numeric_cleaned
0,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,['It was a time honored tradition among the ea...,You Are Special Today Red Plate [With Red Pen],NaN,Waechtersbach USA,[],"['>#39,665 in Kitchen & Dining (See Top 100 in...",Amazon Home,$37.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"['Home & Kitchen', 'Home Dcor', 'Candles & Hol...",NaN,['VICKS INHALER relieves stuffy noses helps si...,Vicks Inhaler Relief for Cold Sinus Nasal Cong...,NaN,Vicks,[],"['>#1,763,185 in Home & Kitchen (See Top 100 i...",Amazon Home,$4.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,"['16 oz squeeze bottle, 1 lb.']",Artistic Churchware Communion Cup Filler: RW525,NaN,Artistic Churchware,"['Religious Supply Center', 'RW-525', 'Communi...","['>#2,127,003 in Home & Kitchen (See Top 100 i...",Amazon Home,$12.48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# --- Comments / reviews (one row per interaction) ---
review_cols = [
    "reviewerID", "asin", "overall",
    "verified", "unixReviewTime", "reviewTime", "vote",
]
df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    usecols=review_cols,
)
print("df_reviews:", df_reviews.shape)
df_reviews.head(3)

/var/folders/86/_khp3pb10vg5vtbr6fmndrxc0000gn/T/ipykernel_31795/645899721.py:6: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv(


df_reviews: (6898955, 7)


,overall,verified,reviewTime,reviewerID,asin,unixReviewTime,vote
0,5.0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600,NaN
1,3.0,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800,2
2,5.0,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800,NaN


In [6]:
# --- Connect the two on `asin` ---
# A curated subset of item-feature columns to attach to each interaction.
feature_cols = [
    "asin", "title_cleaned", "brand",
    "cat_2", "cat_3", "cat_4",
    "Product_Type", "Material", "Color", "extracted_features",
]
feature_cols = [c for c in feature_cols if c in df_features.columns]

df = df_reviews.merge(
    df_features[feature_cols],
    on="asin",
    how="left",
    validate="many_to_one",  # many reviews -> one item row
)

n_unmatched = df["title_cleaned"].isna().sum()
print(f"merged: {df.shape}")
print(f"unique users: {df['reviewerID'].nunique():,} | "
      f"unique items: {df['asin'].nunique():,}")
print(f"reviews with no matching item features: {n_unmatched:,}")
df.head(3)

merged: (6898955, 16)
unique users: 777,242 | unique items: 189,172
reviews with no matching item features: 1,134,069


,overall,verified,reviewTime,reviewerID,asin,unixReviewTime,vote,title_cleaned,brand,cat_2,cat_3,cat_4,Product_Type,Material,Color,extracted_features
0,5.0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600,NaN,welland chicago wall floating corner shelf 20 ...,WELLAND,Home Dcor,Home Dcor Accents,Corner Shelves,corner shelf,None,black,"{'Features': 'floating shelf', 'Dimensions': '..."
1,3.0,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800,2,welland chicago wall floating corner shelf 20 ...,WELLAND,Home Dcor,Home Dcor Accents,Corner Shelves,corner shelf,None,black,"{'Features': 'floating shelf', 'Dimensions': '..."
2,5.0,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800,NaN,stainless coffee mug,Timolino,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,mug,stainless,None,"{'Material': 'stainless', 'Product_Type': 'mug..."
